# EasyGen Pro — MODEL6: Prompt-Based Background Replace

**Runtime → Change runtime type → T4 GPU** before running!

| Mode | Model | Uses prompt? |
|------|-------|----------------|
| 🌄 **Background Replace** (`mode=bg`) | **Stable Diffusion Inpainting** | ✅ Yes — forest, beach, studio, etc. |
| 🗑️ Object removal (`mode=remove`) | LaMa (optional) | No |

1. Run **Cell 1** → **Runtime → Restart session**
2. Run **Cells 2 → 6**
3. Copy ngrok URL into `app.html` → `const BASE_URL_BG = "..."`
4. In the web app: paint subject (green), type prompt, wait ~30–60s

In [ ]:
# ── CELL 1: Install (then Runtime → Restart session) ─────────────────
!pip install -q diffusers transformers accelerate safetensors
!pip install -q opencv-python-headless flask flask-cors pyngrok pillow

import torch
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — enable T4 GPU!")
print("\n>>> Runtime → Restart session, then run Cells 2–6 <<<")

In [ ]:
# ── CELL 2: ngrok token ───────────────────────────────────────────────
from pyngrok import ngrok

ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")  # https://dashboard.ngrok.com/get-started/your-authtoken
print("ngrok token set")

In [ ]:
# ── CELL 3: Load Stable Diffusion Inpainting (prompt-based) ───────────
import io
import threading

import cv2
import numpy as np
import torch
from PIL import Image
from diffusers import StableDiffusionInpaintPipeline
from flask import Flask, jsonify, request, send_file
from flask_cors import CORS
from pyngrok import ngrok

SD_MODEL = "runwayml/stable-diffusion-inpainting"  # reliable on T4; reads your prompt

print(f"Loading {SD_MODEL} (~4 GB first time)...")
sd_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    SD_MODEL,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False,
)
sd_pipe = sd_pipe.to("cuda")
sd_pipe.enable_attention_slicing()
print("SD Inpainting ready on GPU")

In [ ]:
# ── CELL 4: Mask helpers + SD background generator ─────────────────────

def prepare_background_mask(image_pil, mask_pil, blur_radius=8):
    """User paints SUBJECT white → invert → white = area SD repaints."""
    m = mask_pil.resize(image_pil.size, Image.LANCZOS).convert("L")
    arr = np.array(m)
    arr = 255 - ((arr > 128).astype(np.uint8) * 255)
    arr = cv2.dilate(arr, np.ones((5, 5), np.uint8), iterations=2)
    if blur_radius > 0:
        k = blur_radius * 2 + 1
        arr = cv2.GaussianBlur(arr, (k, k), blur_radius)
    return Image.fromarray(arr.astype(np.uint8))


def run_sd_background(image_pil, mask_pil, prompt, steps=35, guidance=7.5, blur=8):
    """Generate new background from text prompt (forest, beach, etc.)."""
    mask = prepare_background_mask(image_pil, mask_pil, blur_radius=blur)
    w, h = image_pil.size
    max_side = 768
    scale = min(1.0, max_side / max(w, h))
    nw = max(64, int(w * scale) // 8 * 8)
    nh = max(64, int(h * scale) // 8 * 8)
    img = image_pil.resize((nw, nh), Image.LANCZOS)
    msk = mask.resize((nw, nh), Image.LANCZOS)

    neg = "blurry, low quality, distorted, deformed, watermark, text, bad anatomy, extra limbs"
    full_prompt = prompt.strip()
    if "background" not in full_prompt.lower():
        full_prompt += ", detailed background, photorealistic, 8k, sharp focus"

    with torch.inference_mode():
        out = sd_pipe(
            prompt=full_prompt,
            negative_prompt=neg,
            image=img,
            mask_image=msk,
            num_inference_steps=int(steps),
            guidance_scale=float(guidance),
        ).images[0]

    return out.resize(image_pil.size, Image.LANCZOS)

print("Helpers ready")

In [ ]:
# ── CELL 5: Flask API + ngrok ─────────────────────────────────────────
app = Flask(__name__)
CORS(app)


@app.route("/health")
def health():
    return jsonify({
        "status": "ok",
        "model": SD_MODEL,
        "background_engine": "stable-diffusion-inpainting",
        "prompt_based": True,
    })


@app.route("/api/inpaint", methods=["POST"])
def inpaint():
    try:
        if "image" not in request.files:
            return jsonify({"error": "No image"}), 400
        if "mask" not in request.files:
            return jsonify({"error": "No mask"}), 400

        mode = request.form.get("mode", "bg").strip().lower()
        prompt = request.form.get("prompt", "").strip()
        neg_prompt = request.form.get("neg_prompt", "").strip()
        mask_blur = max(0, min(20, int(request.form.get("mask_blur", 8))))
        steps = max(15, min(60, int(request.form.get("num_steps", 35))))
        guidance = max(1.0, min(20.0, float(request.form.get("guidance_scale", 7.5))))

        image_pil = Image.open(io.BytesIO(request.files["image"].read())).convert("RGB")
        mask_pil = Image.open(io.BytesIO(request.files["mask"].read())).convert("L")

        print(f"\nmode={mode} | size={image_pil.size} | steps={steps} | prompt={prompt[:80]!r}")

        if mode == "bg":
            if not prompt:
                return jsonify({"error": "Prompt required (e.g. lush green forest background)"}), 400
            result_pil = run_sd_background(
                image_pil, mask_pil, prompt,
                steps=steps, guidance=guidance, blur=mask_blur,
            )
        elif mode == "custom":
            if not prompt:
                return jsonify({"error": "Prompt required for custom fill"}), 400
            # custom = paint area to change (white on mask, no invert)
            m = mask_pil.resize(image_pil.size, Image.LANCZOS).convert("L")
            arr = np.array(m)
            arr = ((arr > 128).astype(np.uint8) * 255)
            if mask_blur > 0:
                k = mask_blur * 2 + 1
                arr = cv2.GaussianBlur(arr, (k, k), mask_blur)
            custom_mask = Image.fromarray(arr.astype(np.uint8))
            w, h = image_pil.size
            nw = max(64, int(min(768, w) // 8 * 8))
            nh = max(64, int(min(768, h) // 8 * 8))
            img = image_pil.resize((nw, nh), Image.LANCZOS)
            msk = custom_mask.resize((nw, nh), Image.LANCZOS)
            neg = neg_prompt or "blurry, low quality, distorted, watermark"
            with torch.inference_mode():
                out = sd_pipe(
                    prompt=prompt,
                    negative_prompt=neg,
                    image=img,
                    mask_image=msk,
                    num_inference_steps=steps,
                    guidance_scale=guidance,
                ).images[0]
            result_pil = out.resize(image_pil.size, Image.LANCZOS)
        else:
            return jsonify({"error": "Use mode=bg for Change Background in app.html"}), 400

        buf = io.BytesIO()
        result_pil.save(buf, format="PNG")
        buf.seek(0)
        print("Done")
        return send_file(buf, mimetype="image/png")

    except Exception as e:
        import traceback
        traceback.print_exc()
        return jsonify({"error": str(e)}), 500


threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=5001, debug=False, use_reloader=False),
    daemon=True,
).start()

public_url = ngrok.connect(5001).public_url
print("\n" + "=" * 60)
print("Prompt-based background backend LIVE:")
print(f"    {public_url}")
print("=" * 60)
print(f'\napp.html → const BASE_URL_BG = "{public_url}";')
print("\nWeb app: paint subject (green) + prompt e.g. forest background")

In [ ]:
# ── CELL 6 (optional): Quick test without web app ─────────────────────
# img = Image.open("/content/your_photo.jpg").convert("RGB")
# mask = Image.new("L", img.size, 0)
# from PIL import ImageDraw
# ImageDraw.Draw(mask).ellipse([100, 50, 400, 500], fill=255)
# out = run_sd_background(img, mask, "lush green forest background, sunlight, 8k")
# display(out)